# 03 · Bronze → Silver

Parse kiểu → tra dimension (LEFT JOIN) → gán **một** `rule_id` cho dòng vi phạm → dòng sạch vào
`fact_valuation`, dòng lỗi vào `quarantine` → đối soát `fact + quarantine = Bronze active`.

In [1]:
import csv, json, logging, shutil, sys
from pathlib import Path

import pandas as pd

# Thư mục gốc dự án = nơi có src/config.py (kernel có thể mở với cwd bất kỳ)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
for p in (ROOT, ROOT / "tests"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src import bronze, config, gold, silver

logging.basicConfig(level=logging.INFO, format="%(levelname)-5s %(name)-7s %(message)s",
                    stream=sys.stdout, force=True)
logging.getLogger("py4j").setLevel(logging.WARNING)
pd.set_option("display.max_columns", 40)

SANDBOX = ROOT / "notebook" / "_sandbox"             # mọi thứ notebook ghi ra nằm ở đây
SB_BRONZE = SANDBOX / "bronze" / "business_valuation"
SB_CONTROL = SANDBOX / "control" / "loaded_batches.csv"
SB_SILVER = SANDBOX / "silver"
REF_COPY = SANDBOX / "ref_model_map_bo_sung.csv"     # bản sao ref đã bổ sung (mục 3.8)

# Số đo trên 18 lô thật trong tài liệu — để SO, không assert (dữ liệu đổi thì số đổi)
EXPECTED = {
    "bronze": 52_791, "fact": 44_748, "quarantine": 8_043,
    "rules": {"CORRUPT_RECORD": 0, "DUP_KEY": 0, "MISSING_FIELD": 34, "DATE_INVALID": 225,
              "OUT_OF_SCOPE": 0, "MODEL_CODE_UNKNOWN": 5_154, "VALUE_INVALID": 280,
              "PRICE_INVALID": 2_350},
}

Hàm tiện ích dùng trong notebook này:

In [2]:
def fresh(*paths: Path) -> None:
    """Xoá các thư mục/file con của sandbox — notebook chỉ dọn phần của mình."""
    for p in paths:
        if SANDBOX not in p.parents:
            raise ValueError(f"{p} nằm ngoài sandbox — không xoá")
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists():
            p.unlink()
    SANDBOX.mkdir(parents=True, exist_ok=True)


def show(df, n: int = 10) -> pd.DataFrame:
    """DataFrame Spark -> pandas để notebook hiển thị thành bảng."""
    return df.limit(n).toPandas()


def compare(actual: dict, expected: dict) -> pd.DataFrame:
    """Bảng so số thực tế với số đo ghi trong tài liệu."""
    keys = list(expected) + [k for k in actual if k not in expected]
    return pd.DataFrame([{"mục": k, "thực tế": actual.get(k, 0), "tài liệu": expected.get(k, "—"),
                          "khớp": actual.get(k, 0) == expected.get(k)} for k in keys])


def ensure_bronze(spark) -> None:
    """Sandbox chưa có Bronze (chưa chạy notebook 02) -> nạp 18 lô."""
    if not SB_BRONZE.exists():
        print("Sandbox chưa có Bronze → nạp 18 lô (như notebook 02)")
        bronze.run(spark, config.LANDING_DIR, SB_BRONZE, SB_CONTROL)


fresh(SB_SILVER, SANDBOX / "silver_bo_sung", REF_COPY)

In [3]:
from pyspark.sql import functions as F
from src.spark import get_spark

spark = get_spark("03_silver")
print("Spark", spark.version)

ensure_bronze(spark)

Spark 3.5.9


## 3.1 Chọn lô active: mỗi tháng lấy bản nạp mới nhất

Nếu notebook 02 đã chạy bước gửi lại lô, tháng `2026-06` có **2 bản nạp** trong Bronze nhưng chỉ **1**
bản active.

In [ ]:
bz = spark.read.parquet(SB_BRONZE.as_posix())
active = silver.active_bronze(bz).cache()
print(f"Bronze: {bz.count()} dòng · active: {active.count()} dòng")

per_month = (bz.groupBy("as_of_month").agg(F.countDistinct("_batch_id").alias("so_ban_nap"),
                                           F.count("*").alias("dong_bronze"))
               .join(active.groupBy("as_of_month").agg(F.first("_batch_id").alias("ban_active"),
                                                       F.countDistinct("_batch_id").alias("n_active"),
                                                       F.count("*").alias("dong_active")), "as_of_month")
               .orderBy("as_of_month").toPandas())

assert (per_month["n_active"] == 1).all(), "Mỗi tháng chỉ được có 1 bản active"
per_month.drop(columns="n_active")

## 3.2 Dựng dimension từ `ref_*` — khoá thay thế tất định

In [ ]:
dim_model = silver.build_dim_model(silver.read_ref_csv(spark, config.REF_MODEL_MAP)).cache()
dim_vehicle_type = silver.build_dim_vehicle_type(silver.read_ref_csv(spark, config.REF_VEH_TYPE)).cache()
print("dim_model:", dim_model.count(), "dòng · dim_vehicle_type:", dim_vehicle_type.count(), "dòng")

# Băm khoá tự nhiên: dựng lại lần hai phải ra ĐÚNG các khoá cũ
again = silver.build_dim_model(silver.read_ref_csv(spark, config.REF_MODEL_MAP))
assert dim_model.select("ma_dong_xe", "model_key").subtract(again.select("ma_dong_xe", "model_key")).count() == 0

display(show(dim_model, 5))
show(dim_vehicle_type, 30)

## 3.3 Từng luật trên dòng dựng tay

Mỗi dòng là một dòng hợp lệ bị đổi vài ô. Nhãn tình huống đặt ở cột `_batch_id` để theo dõi dòng đi
về đâu. Dimension là dimension **thật** ở 3.2.

In [ ]:
from helpers import VALID, bronze_df

ma_xe = dim_model.first()["ma_dong_xe"]                               # mã xe có thật trong dim
nhom = dim_vehicle_type.filter("trong_pham_vi").first()["nhom_xe"]  # loại xe trong phạm vi
base = {**VALID, "ma_dong_xe": ma_xe, "nhom_xe": nhom}

# (tình huống, ma_dinh_gia, ô đổi so với dòng hợp lệ, rule_id mong đợi; None = vào fact)
CASE_LIST = [
    ("hợp lệ",                               "BV-OK",     {},                                      None),
    ("vỡ cột",                               "BV-CORRUPT", {},                                     "CORRUPT_RECORD"),
    ("khoá rỗng",                            "",          {},                                      "DUP_KEY"),
    ("trùng khoá BV-OK, hiệu lực sớm hơn",   "BV-OK",     {"hieu_luc_tu": "2026-06-11"},           "DUP_KEY"),
    ("thiếu số chỗ (toàn khoảng trắng)",     "BV-MISS",   {"so_cho_ngoi": "  "},                   "MISSING_FIELD"),
    ("ngày duyệt sai dạng",                  "BV-DATE1",  {"ngay_duyet": "10/06/2026"},            "DATE_INVALID"),
    ("hiệu lực đến < từ",                    "BV-DATE2",  {"hieu_luc_den": "2026-01-01"},          "DATE_INVALID"),
    ("hiệu lực lệch ngày duyệt > 366 ngày",  "BV-DATE3",  {"hieu_luc_tu": "2024-06-12",
                                                           "hieu_luc_den": "2025-06-12"},          "DATE_INVALID"),
    ("loại xe không có trong dim",           "BV-SCOPE",  {"nhom_xe": "KHONG_CO"},                 "OUT_OF_SCOPE"),
    ("mã xe không có trong dim",             "BV-MODEL",  {"ma_dong_xe": "ZZZ99999"},              "MODEL_CODE_UNKNOWN"),
    ("đời xe > năm duyệt + 1",               "BV-VAL1",   {"nam_san_xuat": "2030"},                "VALUE_INVALID"),
    ("năm sản xuất không phải số",           "BV-VAL2",   {"nam_san_xuat": "abc"},                 "VALUE_INVALID"),
    ("loai_dinh_gia lạ",                     "BV-VAL3",   {"loai_dinh_gia": "demo"},               "VALUE_INVALID"),
    ("giá 0",                                "BV-PRICE1", {"gia_tri_dinh_gia": "0"},               "PRICE_INVALID"),
    ("giá viết kiểu 1.669.000.000",          "BV-PRICE2", {"gia_tri_dinh_gia": "1.669.000.000"},   "PRICE_INVALID"),
    ("mã xe lạ + giá 0 → luật đứng trước",   "BV-PRIO",   {"ma_dong_xe": "ZZZ99999",
                                                           "gia_tri_dinh_gia": "0"},               "MODEL_CODE_UNKNOWN"),
]
out_of_scope = dim_vehicle_type.filter(~F.col("trong_pham_vi")).first()
if out_of_scope:                                                     # loại xe có trong dim nhưng ngoài phạm vi
    CASE_LIST.append(("loại xe trong dim nhưng ngoài phạm vi", "BV-SCOPE2",
                      {"nhom_xe": out_of_scope["nhom_xe"]}, "OUT_OF_SCOPE"))

rows_t = [{**base, **changes, "ma_dinh_gia": key, "_batch_id": case} for case, key, changes, _ in CASE_LIST]
df_cases = bronze_df(spark, rows_t).withColumn(
    bronze.CORRUPT_COL, F.when(F.col("_batch_id") == "vỡ cột", F.lit("dòng,thô,bị,vỡ")))

fact_t, quar_t = silver.transform(df_cases, dim_model, dim_vehicle_type)
got = {r["_batch_id"]: r["rule_id"] for r in quar_t.select("_batch_id", "rule_id").collect()}
in_fact = {r["_batch_id"] for r in fact_t.select("_batch_id").collect()}

table = pd.DataFrame([{"tình huống": case, "ma_dinh_gia": key, "mong đợi": exp or "→ fact",
                       "thực tế": got.get(case) or ("→ fact" if case in in_fact else "MẤT DÒNG")}
                      for case, key, _, exp in CASE_LIST])
table["khớp"] = table["mong đợi"] == table["thực tế"]

assert table["khớp"].all(), "Có dòng đi sai chỗ"
assert fact_t.count() + quar_t.count() == df_cases.count(), "Có dòng biến mất"
table

## 3.4 Chạy luật trên dữ liệu thật (chưa ghi)

In [ ]:
staged = silver.assign_rule_id(silver.parse_and_lookup(active, dim_model, dim_vehicle_type)).cache()
by_rule = {r["rule_id"]: r["count"] for r in staged.groupBy("rule_id").count().collect()}
n_clean = by_rule.pop(None, 0)
print(f"fact {n_clean} · quarantine {sum(by_rule.values())} "
      f"(tài liệu: fact {EXPECTED['fact']}, quarantine {EXPECTED['quarantine']})")
compare(by_rule, EXPECTED["rules"])

## 3.5 "Một dòng — một `rule_id`" nhìn bằng số

Cột trái đếm **độc lập** mỗi luật (một dòng dính 3 luật được đếm 3 lần). Cột phải là sau khi áp thứ
tự ưu tiên. Ví dụ: số dòng giá sai lớn hơn nhiều so với số dòng mang `PRICE_INVALID`, vì phần lớn đã
bị một luật đứng trước bắt.

In [ ]:
parsed = silver.parse_and_lookup(active, dim_model, dim_vehicle_type)
flags = parsed.select([c.alias(r) for r, c in silver.rule_conditions(parsed).items()])
viol = flags.select([F.count_if(F.col(r)).alias(r) for r in silver.RULES]).first().asDict()
pd.DataFrame({"vi phạm (đếm độc lập)": viol,
              "mang rule_id (sau ưu tiên)": {r: by_rule.get(r, 0) for r in silver.RULES}})

## 3.6 Ghi Silver

In [ ]:
res_silver = silver.run(spark, SB_BRONZE, SB_SILVER)
compare(res_silver, {"bronze": EXPECTED["bronze"], "fact": EXPECTED["fact"],
                     "quarantine": EXPECTED["quarantine"]})

## 3.7 Kiểm bất biến trên các bảng đã ghi

In [ ]:
T = {name: spark.read.parquet((SB_SILVER / name).as_posix()) for name in gold.SILVER_TABLES}
fact, quar, dim_date = T["fact_valuation"], T["quarantine"], T["dim_date"]


def orphans(key: str, dim, dim_key: str) -> int:
    """Số dòng fact có khoá ngoại không tìm thấy trong dimension."""
    return fact.join(dim.withColumnRenamed(dim_key, key), key, "left_anti").count()


checks = pd.DataFrame([
    ("fact + quarantine = Bronze active", fact.count() + quar.count() == active.count()),
    ("ma_dinh_gia duy nhất trong fact", fact.select("ma_dinh_gia").distinct().count() == fact.count()),
    ("mọi model_key có trong dim_model", orphans("model_key", T["dim_model"], "model_key") == 0),
    ("mọi vehicle_type_key có trong dim_vehicle_type",
     orphans("vehicle_type_key", T["dim_vehicle_type"], "vehicle_type_key") == 0),
    ("mọi ngay_duyet_key có trong dim_date", orphans("ngay_duyet_key", dim_date, "date_key") == 0),
    ("mọi hieu_luc_tu_key có trong dim_date", orphans("hieu_luc_tu_key", dim_date, "date_key") == 0),
    ("quarantine = cột thô Bronze + rule_id", set(quar.columns) == set(active.columns) | {"rule_id"}),
    ("mọi dòng quarantine có rule_id", quar.filter(F.col("rule_id").isNull()).count() == 0),
], columns=["bất biến", "đạt"])

assert checks["đạt"].all()
fact.printSchema()
checks

In [ ]:
display(show(fact, 5))
show(quar.groupBy("rule_id").count().orderBy(F.desc("count")))

## 3.8 Cứu dòng quarantine: bổ sung `ref_model_map` rồi chạy lại Silver

Lấy mã xe chưa có trong bảng ref xuất hiện nhiều nhất, thêm vào **bản sao** của `ref_model_map.csv`
(hãng/dòng xe là giá trị thử), chạy lại Silver sang thư mục khác. Không sửa code, không cần bên cung
cấp gửi lại lô.

In [ ]:
top = (quar.filter("rule_id = 'MODEL_CODE_UNKNOWN'").groupBy("ma_dong_xe").count()
          .orderBy(F.desc("count")).first())
print(f"Mã chưa có trong ref nhiều nhất: {top['ma_dong_xe']!r} — {top['count']} dòng")

with open(config.REF_MODEL_MAP, newline="", encoding="utf-8-sig") as f:
    ref_rows = list(csv.DictReader(f))
ref_rows.append({"ma_dong_xe": top["ma_dong_xe"].strip(), "ma_hieu": "THU01",
                 "hang_xe": "(hãng thử)", "hieu_xe": "(dòng thử)"})
with open(REF_COPY, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(ref_rows[0]))
    w.writeheader()
    w.writerows(ref_rows)

res_rescue = silver.run(spark, SB_BRONZE, SANDBOX / "silver_bo_sung", ref_model_map=REF_COPY)
q2 = spark.read.parquet((SANDBOX / "silver_bo_sung" / "quarantine").as_posix())
print(f"fact {res_silver['fact']} → {res_rescue['fact']}  (+{res_rescue['fact'] - res_silver['fact']})")

same_code = F.col("ma_dong_xe") == top["ma_dong_xe"]
pd.DataFrame({"trước": {r["rule_id"]: r["count"] for r in quar.filter(same_code).groupBy("rule_id").count().collect()},
              "sau bổ sung": {r["rule_id"]: r["count"] for r in q2.filter(same_code).groupBy("rule_id").count().collect()}}).fillna(0).astype(int)

---
## Dọn dẹp

In [ ]:
spark.stop()
# fresh(SANDBOX / ...)    # xoá phần sandbox của notebook này nếu muốn